In [3]:
import sys
import json
import time
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize_scalar

MODE = "cross_ea"          # "cross" or "cross_ea"
N_SAMPLES = 300

# Works both as a script and in a notebook sitting in the optimization folder
try:
    HERE = Path(__file__).resolve().parent
    IN_NOTEBOOK = False
except NameError:
    HERE = Path(".").resolve()
    IN_NOTEBOOK = True

sys.path.insert(0, str(HERE.parent))          # wirecoating.py is one folder up
from wirecoating import WireCoatingDie, CrossLayer

# Constants and polymers (same as line_speed_optimization.ipynb)
K0 = 273.15
R_W, R_D, L_LAND = 0.577e-3 / 2, 1.25e-3 / 2, 0.10e-3
TAU_C = 2e5                       # critical wall shear stress [Pa]
T_FEP_MIN, T_TOTAL_MAX = 0.20e-3, 0.25e-3
T = 330 + K0

FEP = CrossLayer(eta0=3040, lam=0.0103, m=0.686, E_a=51.4e3, T_ref=330 + K0,
                 name="FEP 9835", rho_melt=1400, rho_solid=2100)
HT = CrossLayer(eta0=12780, lam=1.0345, m=0.529, E_a=92.4e3, T_ref=300 + K0,
                name="HT 2181", rho_melt=1400, rho_solid=1700)

EA_SE = {"FEP": 2.8e3, "HT": 9.7e3}    # activation energy standard errors from Table 1 [J/mol]


def make_die(V, layers):
    return WireCoatingDie(R_W, R_D, V, list(layers), L=L_LAND)


def speed_limit(thicknesses, layers):
    """Line speed V* [m/s] where the larger wall shear stress hits TAU_C."""
    state = {"warm": None}

    def g(lnV):
        sol = make_die(np.exp(lnV), layers).solve(thicknesses=thicknesses, T=T,
                                                  warm_start=state["warm"])
        state["warm"] = sol
        return max(abs(sol.tau_wire), abs(sol.tau_die)) / TAU_C - 1.0

    return float(np.exp(brentq(g, np.log(1e-4), np.log(20.0), xtol=1e-10)))


def best_ht(fep, ht):
    """Optimal HT thickness [mm] and the speed limit V* [m/s] it gives."""
    upper = T_TOTAL_MAX - T_FEP_MIN
    res = minimize_scalar(lambda x: -speed_limit([T_FEP_MIN, x * 1e-3], (fep, ht)),
                          bounds=(0.002, upper * 1e3), method="bounded", options={"xatol": 1e-5})
    return res.x, -res.fun


COV = json.load(open(HERE / "cross_covariance.json"))


def draw(rng, key):
    """One random (eta0, lam, m) set from the fit's covariance (fit was in ln eta0, ln lam, m)."""
    c = COV[key]
    mu = np.array([c["best"][v] for v in c["vars"]])
    le, ll, m = rng.multivariate_normal(mu, np.array(c["covar"]))
    return dict(eta0=np.exp(le), lam=np.exp(ll), m=m)


def run(job):
    i, fep_kw, ht_kw = job
    try:
        t, V = best_ht(replace(FEP, **fep_kw), replace(HT, **ht_kw))
        return dict(i=i, **{f"FEP_{k}": v for k, v in fep_kw.items()},
                    **{f"HT_{k}": v for k, v in ht_kw.items()}, t_HT_mm=t, V=V, ok=True)
    except Exception as e:
        return dict(i=i, ok=False, err=str(e)[:80])


def make_jobs(mode, n):
    rng = np.random.default_rng(431)     # fixed seed so results are repeatable
    jobs = []
    for i in range(n):
        f, h = draw(rng, "FEP"), draw(rng, "HT 2181")
        if mode == "cross_ea":
            f["E_a"] = FEP.E_a + EA_SE["FEP"] * rng.standard_normal()
            h["E_a"] = HT.E_a + EA_SE["HT"] * rng.standard_normal()
        jobs.append((i, f, h))
    return jobs


def summarize(out, mode, seconds):
    out.to_csv(HERE / f"sensitivity_mc_{mode}.csv", index=False)
    ok = out[out.ok]
    print(f"\n{mode}: {len(ok)}/{len(out)} samples ok, {seconds:.0f} s")
    for col, unit in [("t_HT_mm", "mm"), ("V", "m/s")]:
        lo, med, hi = ok[col].quantile([0.025, 0.5, 0.975]).values
        print(f"  {col}: median {med:.4f} {unit}, 95% interval [{lo:.4f}, {hi:.4f}] {unit}")
    print(f"  saved every sample to sensitivity_mc_{mode}.csv")


if __name__ == "__main__":
    if IN_NOTEBOOK:
        mode, n = MODE, N_SAMPLES
    else:
        mode, n = sys.argv[1], int(sys.argv[2])
    jobs = make_jobs(mode, n)
    t0 = time.time()
    if IN_NOTEBOOK:
        rows = []
        for k, job in enumerate(jobs, 1):
            rows.append(run(job))
            if k % 25 == 0:
                print(f"  {k}/{n} done, {time.time() - t0:.0f} s")
        out = pd.DataFrame(rows)
    else:
        from multiprocessing import Pool
        with Pool(2) as p:
            out = pd.DataFrame(p.map(run, jobs, chunksize=4))
    summarize(out, mode, time.time() - t0)

  25/300 done, 66 s
  50/300 done, 131 s
  75/300 done, 195 s
  100/300 done, 260 s
  125/300 done, 326 s
  150/300 done, 393 s
  175/300 done, 459 s
  200/300 done, 525 s
  225/300 done, 590 s
  250/300 done, 656 s
  275/300 done, 723 s
  300/300 done, 787 s

cross_ea: 300/300 samples ok, 787 s
  t_HT_mm: median 0.0284 mm, 95% interval [0.0279, 0.0288] mm
  V: median 0.0568 m/s, 95% interval [0.0477, 0.0683] m/s
  saved every sample to sensitivity_mc_cross_ea.csv
